# Data preparation

In [ ]:
import os
import pandas as pd   

BASE_DIR = os.path.abspath("..")  # go from notebooks → project root

REAL_PATH = os.path.join(BASE_DIR, "data", "raw", "EU_C12N15_data.csv")
SAMPLE_PATH = os.path.join(BASE_DIR, "data", "sample.csv")

if os.path.exists(REAL_PATH):
    DATA_PATH = REAL_PATH
elif os.path.exists(SAMPLE_PATH):
    DATA_PATH = SAMPLE_PATH
else:
    raise FileNotFoundError(
        f"No dataset found.\nChecked:\n- {REAL_PATH}\n- {SAMPLE_PATH}"
    )

raw_data = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", raw_data.shape)

✅ Loaded: /Users/carlos38/Desktop/Profesional/Master/GNN_course/EU_collabo_project/data/raw/EU_C12N15_data.csv
Shape: (3180, 14)


In [175]:
import pandas as pd
import ast

def parse_list_column(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return [i.strip() for i in str(x).split(";")]

In [187]:
df = raw_data[raw_data["Publication Year"] < 2025].copy()
df["Applicants"] = df["Applicants"].apply(parse_list_column)
df["Inventors"] = df["Inventors"].apply(parse_list_column)
df["CPC Classifications"] = df["CPC Classifications"].apply(parse_list_column)
df["Priority Numbers"] = df["Priority Numbers"].apply(parse_list_column)

In [ ]:
from rapidfuzz import fuzz

TOKEN_REPLACEMENTS = {
    "biotechnologies": "biotechnology",
    "technologies": "technology",
    "therapeutics": "therapeutic",
    "laboratories": "laboratory",
    "systems": "system",
    "sciences": "science",
    "holdings": "holding",
    "communications": "communication",
    "solutions": "solution",
}

def normalize_applicant_name(name):
    if pd.isna(name):
        return ""
    
    name = str(name).lower().strip()
    
    # basic cleanup
    name = name.replace("&", " and ")
    name = name.replace(",", " ")
    name = name.replace(".", " ")
    name = name.replace("  ", " ")
    
    # collapse repeated spaces
    name = " ".join(name.split())

    # token-level replacements
    tokens = [TOKEN_REPLACEMENTS.get(tok, tok) for tok in name.split()]
    name = " ".join(tokens)
    
    return name


def fuzzy_merge_applicants(applicant_names, threshold=95):
    """
    Returns:
        canonical_map: dict mapping original normalized name -> canonical normalized name
        merge_pairs: list of merged pairs for inspection
    """
    applicant_names = sorted(set(a for a in applicant_names if a))
    canonical_map = {}
    canonicals = []
    merge_pairs = []

    for name in applicant_names:
        matched = False

        for canon in canonicals:
            score = fuzz.ratio(name, canon)

            # extra safeguard: only merge if names are extremely close
            if score >= threshold:
                canonical_map[name] = canon
                merge_pairs.append((name, canon, score))
                matched = True
                break

        if not matched:
            canonical_map[name] = name
            canonicals.append(name)

    return canonical_map, merge_pairs

# CLEAN APPLICANT NAMES
df["Applicants"] = df["Applicants"].apply(
    lambda lst: [normalize_applicant_name(a) for a in lst]
)

all_applicants_raw = sorted(set(
    a for row in df["Applicants"] for a in row if a
))

applicant_name_map, merge_pairs = fuzzy_merge_applicants(
    all_applicants_raw,
    threshold=95
)

df["Applicants"] = df["Applicants"].apply(
    lambda lst: [applicant_name_map.get(a, a) for a in lst]
)

print(f"Unique applicants before fuzzy merge: {len(all_applicants_raw)}")
print(f"Unique applicants after fuzzy merge: {len(set(applicant_name_map.values()))}")
print(f"Number of merges made: {len(merge_pairs)}")

merge_log_df = pd.DataFrame(merge_pairs, columns=["variant", "canonical", "score"])
display(merge_log_df.sort_values(["canonical", "score"], ascending=[True, False]).head(50))

Unique applicants before fuzzy merge: 4516
Unique applicants after fuzzy merge: 4432
Number of merges made: 84


,variant,canonical,score
69,tron—translationale onkologie an der univ der ...,03 tron—translationale onkologie an der univ d...,96.511628
0,academisch ziekenhuis leiden a,academisch ziekenhuis leiden,96.551724
1,academisch ziekenhuis leiden h,academisch ziekenhuis leiden,96.551724
2,adiutide pharmaceuticals gmbh,adiu tide pharmaceuticals gmbh,98.305085
3,ann and robert h lurie childrens hospital of c...,ann and robert h lurie childrens hosp of chicago,96.000000
4,belgian volition srl,belgian volition sprl,97.560976
5,biocartis nv,biocartis n v,96.000000
6,bullerdiek jorn,bullerdiek joern,96.774194
7,cabezon silva teresa elisa vir,cabezon siliva teresa elisa vi,96.666667
8,cemm forschungszentrum fuer molekulare medizin...,cemm - forschungzentrum fuer molekulare medizi...,97.087379


In [189]:
merge_log_df.to_csv(os.path.join(BASE_DIR, "data", "processed", "applicant_name_merges.csv"), index=False)

In [ ]:
# Getting applicants across dataset
all_applicants = sorted(set(
    a for row in df["Applicants"] for a in row
))

# Create mappings
applicant_to_id = {name: i for i, name in enumerate(all_applicants)}
id_to_applicant = {i: name for name, i in applicant_to_id.items()}

In [191]:
nodes_df = pd.DataFrame({
    "node_id": list(id_to_applicant.keys()),
    "name": list(id_to_applicant.values())
})

nodes_df.to_csv("data/processed/metadata/nodes_master.csv", index=False)

In [ ]:
from torch_geometric.data import HeteroData
import torch

def build_pyg_hetero_graph(
    df,
    applicant_to_id,
    inventor_to_id,
    patent_to_id,
    cpc_to_id
):

    data = HeteroData()

    # Building edges (using GLOBAL mappings)
    app_pat_edges = []
    inv_pat_edges = []
    cpc_pat_edges = []

    for _, row in df.iterrows():

        pats = row["Priority Numbers"] or [row["Lens ID"]]

        for p in pats:

            # Skip if not in global mapping
            if p not in patent_to_id:
                continue

            p_id = patent_to_id[p]

            # Applicants → Patent
            for a in row["Applicants"]:
                if a in applicant_to_id:
                    app_pat_edges.append([applicant_to_id[a], p_id])

            # Inventors → Patent
            for i in row["Inventors"]:
                if i in inventor_to_id:
                    inv_pat_edges.append([inventor_to_id[i], p_id])

            # CPC → Patent
            for c in row["CPC Classifications"]:
                if c in cpc_to_id:
                    cpc_pat_edges.append([cpc_to_id[c], p_id])

    # Converting to tensors
    def to_edge_index(edges):
        if len(edges) == 0:
            return torch.empty((2, 0), dtype=torch.long)
        return torch.tensor(edges, dtype=torch.long).t().contiguous()

    # Applicant - Patent
    data["applicant", "files", "patent"].edge_index = to_edge_index(app_pat_edges)
    data["patent", "rev_files", "applicant"].edge_index = to_edge_index(
        [[j, i] for i, j in app_pat_edges]
    )

    # Inventor - Patent
    data["inventor", "invented", "patent"].edge_index = to_edge_index(inv_pat_edges)
    data["patent", "rev_invented", "inventor"].edge_index = to_edge_index(
        [[j, i] for i, j in inv_pat_edges]
    )

    # CPC - Patent
    data["cpc", "in_patent", "patent"].edge_index = to_edge_index(cpc_pat_edges)
    data["patent", "rev_in_patent", "cpc"].edge_index = to_edge_index(
        [[j, i] for i, j in cpc_pat_edges]
    )

    # Node counts (GLOBAL)
    data["applicant"].num_nodes = len(applicant_to_id)
    data["inventor"].num_nodes = len(inventor_to_id)
    data["patent"].num_nodes = len(patent_to_id)
    data["cpc"].num_nodes = len(cpc_to_id)

    return data

In [ ]:
# GLOBAL NODE REGISTRY
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return [x]

# Safety cleaning
df["Applicants"] = df["Applicants"].apply(ensure_list)
df["Inventors"] = df["Inventors"].apply(ensure_list)
df["CPC Classifications"] = df["CPC Classifications"].apply(ensure_list)
df["Priority Numbers"] = df["Priority Numbers"].apply(ensure_list)

# Build global sets
all_applicants = sorted(set(
    a for row in df["Applicants"] for a in row
))
all_inventors = sorted(set(
    i for row in df["Inventors"] for i in row
))
all_patents = sorted(set(
    p for _, row in df.iterrows()
    for p in (row["Priority Numbers"] or [row["Lens ID"]])
))
all_cpcs = sorted(set(
    c for row in df["CPC Classifications"] for c in row
))

# Create mappings
applicant_to_id = {a: i for i, a in enumerate(all_applicants)}
inventor_to_id  = {i: j for j, i in enumerate(all_inventors)}
patent_to_id    = {p: i for i, p in enumerate(all_patents)}
cpc_to_id       = {c: i for i, c in enumerate(all_cpcs)}

In [194]:
data_by_year = {}

years = sorted(df["Publication Year"].dropna().unique())

for year in years:

    df_year = df[df["Publication Year"] == year]

    data_year = build_pyg_hetero_graph(
        df_year,
        applicant_to_id,
        inventor_to_id,
        patent_to_id,
        cpc_to_id
    )

    data_by_year[year] = data_year

In [ ]:
# Check consistency across years
for year, data in data_by_year.items():
    assert data["applicant"].num_nodes == len(applicant_to_id)
    assert data["patent"].num_nodes == len(patent_to_id)

print("All yearly graphs share the same node space")

✅ All yearly graphs share the same node space


# Feature Engineering

In [ ]:
import torch
from torch_geometric.nn import MetaPath2Vec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. MetaPaths (STRUCTURAL SIGNAL ONLY)
METAPATHS = {
    "APA": {
        "path": [
            ("applicant","files","patent"),
            ("patent","rev_files","applicant")
        ],
        "weight": 1.0
    },
    "API": {
        "path": [
            ("applicant","files","patent"),
            ("patent","rev_invented","inventor"),
            ("inventor","invented","patent"),
            ("patent","rev_files","applicant")
        ],
        "weight": 1.2
    },
    "APC": {
        "path": [
            ("applicant","files","patent"),
            ("patent","rev_in_patent","cpc"),
            ("cpc","in_patent","patent"),
            ("patent","rev_files","applicant")
        ],
        "weight": 1.2
    }
}


# 2. MetaPath2Vec training
def train_metapath_embedding(data, metapath, emb_dim=32, epochs=2):

    num_nodes = data["applicant"].num_nodes

    try:
        model = MetaPath2Vec(
            data.edge_index_dict,
            embedding_dim=emb_dim,
            metapath=metapath,
            walk_length=30,
            context_size=5,
            walks_per_node=3,
            num_negative_samples=5,
            sparse=True
        ).to(device)

        optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)
        loader = model.loader(batch_size=128, shuffle=True)

        model.train()

        for _ in range(epochs):
            for pos_rw, neg_rw in loader:

                pos_rw = pos_rw.to(device)
                neg_rw = neg_rw.to(device)

                optimizer.zero_grad()
                loss = model.loss(pos_rw, neg_rw)

                # Safety against instability
                if torch.isnan(loss):
                    continue

                loss.backward()
                optimizer.step()

        model.eval()

        with torch.no_grad():
            emb = model("applicant").cpu()

        emb = torch.nan_to_num(emb, nan=0.0, posinf=5.0, neginf=-5.0)

    except Exception:
        # Fallback if metapath fails
        emb = torch.zeros(num_nodes, emb_dim)

    if emb.shape[0] != num_nodes:
        full_emb = torch.zeros(num_nodes, emb_dim)
        full_emb[:emb.shape[0]] = emb
        emb = full_emb

    return emb


# 3. Normalizing embeddings
def normalize_embeddings(z):
    return z / (z.norm(dim=1, keepdim=True) + 1e-8)


# 4. Generating embeddings per year
def generate_temporal_embeddings(data_by_year, metapaths=METAPATHS):

    emb_by_year = {}

    for year, data in data_by_year.items():

        emb_list = []

        for name, cfg in metapaths.items():

            emb = train_metapath_embedding(data, cfg["path"])

            # Apply importance weighting
            emb = emb * cfg["weight"]

            emb_list.append(emb)

        # Concatenate all metapath embeddings
        z = torch.cat(emb_list, dim=1)

        # Normalize for stability + similarity learning
        z = normalize_embeddings(z)

        emb_by_year[year] = z

    return emb_by_year


# 5. Build temporal tensor (T x N x D)
def build_temporal_tensor(emb_by_year, applicant_to_id):

    years_sorted = sorted(emb_by_year.keys())

    T = len(years_sorted)
    N = len(applicant_to_id)
    D = next(iter(emb_by_year.values())).shape[1]

    X = torch.zeros(T, N, D)

    for t, year in enumerate(years_sorted):
        X[t] = emb_by_year[year]

    return X, years_sorted


# 6. Debug graph structure
for year, data in data_by_year.items():
    print(f"{year}: {list(data.edge_index_dict.keys())}")


# 7. Run feature pipeline
emb_by_year = generate_temporal_embeddings(data_by_year)

X, years_sorted = build_temporal_tensor(emb_by_year, applicant_to_id)

print("✅ Feature tensor shape:", X.shape)  # (T, N, D)

1981: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1983: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1984: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1985: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1986: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'pat

# Model for visualization

In [ ]:
# TEMPORAL GNN + SUPERVISED TEMPORAL LINK PREDICTION + EXPORT
# SNAPSHOT-BASED TEMPORAL GNN VERSION

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.utils import negative_sampling, add_self_loops
from torch_geometric.nn import GCNConv
import json
import os

# GLOBAL PATHS
BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(os.path.join(DATA_DIR, "edges"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "metadata"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "embeddings"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "predictions"), exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

# SEED
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# REQUIRED OBJECTS
required_vars = ["X", "data_by_year", "applicant_to_id"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}")

global_apps = applicant_to_id
num_global_nodes = len(global_apps)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# HELPERS
def canonicalize_edge_index(edge_index: torch.Tensor) -> torch.Tensor:
    if edge_index is None or edge_index.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long)

    arr = edge_index.t().cpu().numpy().astype(np.int64)
    arr = arr[arr[:, 0] != arr[:, 1]]
    if len(arr) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    arr = np.sort(arr, axis=1)
    arr = np.unique(arr, axis=0)

    if len(arr) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    return torch.tensor(arr, dtype=torch.long).t().contiguous()


def to_undirected_edge_index(edge_index: torch.Tensor) -> torch.Tensor:
    if edge_index is None or edge_index.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long)

    rev = edge_index[[1, 0], :]
    out = torch.cat([edge_index, rev], dim=1)
    return canonicalize_edge_index(out)


def get_edge_index_from_store(data, edge_type):
    if edge_type not in data.edge_types:
        return None

    store = data[edge_type]

    if "edge_index" in store and store["edge_index"] is not None:
        return store["edge_index"].long()

    if "edge_label_index" in store and store["edge_label_index"] is not None:
        return store["edge_label_index"].long()

    if "adj_t" in store and store["adj_t"] is not None:
        row, col, _ = store["adj_t"].t().coo()
        return torch.stack([row, col], dim=0).long()

    return None


def find_applicant_patent_edge_types(data):
    files_type = None
    rev_files_type = None

    preferred_forward = []
    preferred_reverse = []
    fallback_forward = []
    fallback_reverse = []

    for et in data.edge_types:
        src, rel, dst = et
        rel_l = str(rel).lower()

        if src == "applicant" and dst == "patent":
            fallback_forward.append(et)
            if "file" in rel_l:
                preferred_forward.append(et)

        if src == "patent" and dst == "applicant":
            fallback_reverse.append(et)
            if "file" in rel_l:
                preferred_reverse.append(et)

    if preferred_forward:
        files_type = preferred_forward[0]
    elif fallback_forward:
        files_type = fallback_forward[0]

    if preferred_reverse:
        rev_files_type = preferred_reverse[0]
    elif fallback_reverse:
        rev_files_type = fallback_reverse[0]

    return files_type, rev_files_type


def infer_year_app_map(year, data, global_apps):
    if "app_maps" in globals() and year in app_maps:
        return app_maps[year]

    num_local = int(data["applicant"].num_nodes)

    candidate_keys = [
        "name", "names", "applicant_name", "applicant_names",
        "label", "labels", "id", "ids"
    ]

    for key in candidate_keys:
        if key in data["applicant"]:
            values = data["applicant"][key]

            if torch.is_tensor(values):
                values = values.cpu().tolist()
            else:
                values = list(values)

            inferred = {}
            ok = True

            for i, v in enumerate(values):
                if isinstance(v, bytes):
                    v = v.decode("utf-8", errors="ignore")
                v = str(v)

                if v not in global_apps:
                    ok = False
                    break

                inferred[v] = i

            if ok and len(inferred) > 0:
                print(f"Using inferred applicant mapping from '{key}' for year {year}")
                return inferred

    if num_local <= len(global_apps):
        print(f"Using identity applicant mapping fallback for year {year}")
        return {i: i for i in range(num_local)}

    raise ValueError(
        f"Could not infer applicant mapping for year {year}. "
        f"Local applicant nodes: {num_local}, global applicants: {len(global_apps)}"
    )


def map_local_applicant_id_to_global(local_id, app_map, global_map):
    if all(isinstance(k, (int, np.integer)) for k in app_map.keys()):
        return app_map.get(local_id, None)

    inv_app_map = {v: k for k, v in app_map.items()}
    applicant_key = inv_app_map.get(local_id, None)
    if applicant_key is None:
        return None
    return global_map.get(applicant_key, None)


def reconstruct_global_collab_edges_from_patents(year, data, app_map, global_map):
    files_type, rev_files_type = find_applicant_patent_edge_types(data)

    if files_type is None and rev_files_type is None:
        raise ValueError(
            f"Could not find applicant-patent relations for year {year}. "
            f"Available edge types: {data.edge_types}"
        )

    app_pat_parts = []

    if files_type is not None:
        ei = get_edge_index_from_store(data, files_type)
        if ei is not None and ei.numel() > 0:
            app_pat_parts.append(ei)

    if rev_files_type is not None:
        ei = get_edge_index_from_store(data, rev_files_type)
        if ei is not None and ei.numel() > 0:
            app_pat_parts.append(torch.stack([ei[1], ei[0]], dim=0))

    if len(app_pat_parts) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    app_pat = torch.cat(app_pat_parts, dim=1)
    app_pat = torch.unique(app_pat.t(), dim=0).t().contiguous()

    patent_to_global_apps = {}

    for local_app_id, patent_id in app_pat.t().tolist():
        global_app_id = map_local_applicant_id_to_global(local_app_id, app_map, global_map)
        if global_app_id is None:
            continue
        if global_app_id < 0 or global_app_id >= len(global_map):
            continue

        patent_to_global_apps.setdefault(patent_id, set()).add(global_app_id)

    collab_edges = set()

    for _, app_set in patent_to_global_apps.items():
        apps = sorted(app_set)
        if len(apps) < 2:
            continue

        for i in range(len(apps)):
            for j in range(i + 1, len(apps)):
                a, b = apps[i], apps[j]
                if a != b:
                    x, y = (a, b) if a < b else (b, a)
                    collab_edges.add((x, y))

    if len(collab_edges) == 0:
        print(f"Year {year}: reconstructed collaboration edges are empty")
        return torch.empty((2, 0), dtype=torch.long)

    arr = np.array(list(collab_edges), dtype=np.int64)
    return torch.tensor(arr, dtype=torch.long).t().contiguous()


def build_yearly_global_graphs(data_by_year, years, global_apps):
    yearly_edges = {}
    for y in years:
        year_app_map = infer_year_app_map(y, data_by_year[y], global_apps)
        edge_index = reconstruct_global_collab_edges_from_patents(
            year=y,
            data=data_by_year[y],
            app_map=year_app_map,
            global_map=global_apps
        )
        edge_index = canonicalize_edge_index(edge_index)
        edge_index = to_undirected_edge_index(edge_index)
        edge_index, _ = add_self_loops(edge_index, num_nodes=len(global_apps))
        yearly_edges[y] = edge_index.long()
        print(f"Year {y}: graph edges = {tuple(yearly_edges[y].shape)}")
    return yearly_edges


# TEMPORAL GNN MODEL
class SnapshotTemporalGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        gnn_hidden_dim=64,
        rnn_hidden_dim=64,
        gnn_layers=2,
        dropout=0.2,
        use_attention=True,
        predictor="mlp"
    ):
        super().__init__()

        self.use_attention = use_attention
        self.predictor = predictor
        self.dropout = dropout

        self.input_proj = nn.Linear(in_dim, gnn_hidden_dim)

        self.gnn_convs = nn.ModuleList()
        self.gnn_convs.append(GCNConv(gnn_hidden_dim, gnn_hidden_dim))
        for _ in range(gnn_layers - 1):
            self.gnn_convs.append(GCNConv(gnn_hidden_dim, gnn_hidden_dim))

        self.gru = nn.GRU(
            input_size=gnn_hidden_dim,
            hidden_size=rnn_hidden_dim,
            num_layers=1
        )

        self.att_proj = nn.Linear(rnn_hidden_dim, 1)

        self.mlp = nn.Sequential(
            nn.Linear(rnn_hidden_dim * 4, rnn_hidden_dim),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Dropout(dropout),
            nn.Linear(rnn_hidden_dim, 1)
        )

    def encode_snapshot(self, x_t, edge_index_t):
        h = self.input_proj(x_t)

        for i, conv in enumerate(self.gnn_convs):
            h = conv(h, edge_index_t)
            if i < len(self.gnn_convs) - 1:
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)

        h = torch.nan_to_num(h, nan=0.0, posinf=5.0, neginf=-5.0)
        return h

    def encode(self, X_seq, edge_index_seq):
        snapshot_embs = []

        for t in range(X_seq.size(0)):
            h_t = self.encode_snapshot(X_seq[t], edge_index_seq[t])
            snapshot_embs.append(h_t)

        H = torch.stack(snapshot_embs, dim=0)   # [T, N, H_gnn]
        H_temporal, _ = self.gru(H)             # [T, N, H_rnn]

        if self.use_attention:
            att = torch.clamp(self.att_proj(H_temporal), -10, 10)
            scores = torch.softmax(att, dim=0)
            z = (H_temporal * scores).sum(dim=0)
        else:
            z = H_temporal.mean(dim=0)

        z = torch.nan_to_num(z, nan=0.0, posinf=5.0, neginf=-5.0)
        return H_temporal, z

    def decode_logits(self, z, edge_index):
        if edge_index.numel() == 0:
            return torch.empty(0, device=z.device)

        src, dst = edge_index
        s = z[src]
        d = z[dst]

        if self.predictor == "dot":
            return (s * d).sum(dim=1)

        feat = torch.cat([
            s,
            d,
            torch.abs(s - d),
            s * d
        ], dim=1)

        return self.mlp(feat).view(-1)

    def decode_proba(self, z, edge_index, clip=8.0):
        logits = self.decode_logits(z, edge_index)
        logits = torch.clamp(logits, -clip, clip)
        return torch.sigmoid(logits)


# TEMPORAL SPLIT
years = sorted(data_by_year.keys())
if len(years) < 2:
    raise ValueError("Need at least two years for temporal split.")

train_years = years[:-1]
test_year = years[-1]
train_T = len(train_years)

print("Train years:", train_years)
print("Test year:", test_year)

# NORMALIZE FEATURES
X = torch.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0).float()

X_train_window = X[:train_T]
mean = X_train_window.mean(dim=(0, 1), keepdim=True)
std = X_train_window.std(dim=(0, 1), keepdim=True)
std = torch.where(std < 1e-6, torch.ones_like(std), std)

X = (X - mean) / std
X = torch.nan_to_num(X, nan=0.0, posinf=5.0, neginf=-5.0)

# critical fix
X = X.detach().clone()
X_train = X[:train_T].detach().clone().to(device)
print("X_train shape:", tuple(X_train.shape))

# BUILD YEARLY APPLICANT GRAPHS
all_yearly_edges = build_yearly_global_graphs(data_by_year, years, global_apps)

train_edge_index_seq = [all_yearly_edges[y].to(device) for y in train_years]

train_edge_list = []
for y in train_years:
    e = canonicalize_edge_index(all_yearly_edges[y].cpu())
    if e.numel() > 0:
        train_edge_list.append(e)

if len(train_edge_list) == 0:
    raise ValueError("❌ No training collaboration edges were reconstructed from train years.")

train_edge_index = canonicalize_edge_index(torch.cat(train_edge_list, dim=1)).to(device)
if train_edge_index.numel() == 0:
    raise ValueError("❌ Training edge index is empty after cleaning.")

np.save(
    os.path.join(DATA_DIR, "edges", "train_edges.npy"),
    train_edge_index.cpu().numpy()
)
print("✅ Saved train edges:", tuple(train_edge_index.shape))

test_edge_index = canonicalize_edge_index(all_yearly_edges[test_year].cpu()).to(device)
if test_edge_index.numel() == 0:
    raise ValueError("❌ No test collaboration edges were reconstructed for the final year.")

np.save(
    os.path.join(DATA_DIR, "edges", "test_edges.npy"),
    test_edge_index.cpu().numpy()
)
print("✅ Saved test edges:", tuple(test_edge_index.shape))

# MODEL
model = SnapshotTemporalGNN(
    in_dim=X_train.shape[-1],
    gnn_hidden_dim=64,
    rnn_hidden_dim=64,
    gnn_layers=2,
    dropout=0.2,
    use_attention=True,
    predictor="mlp"
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

losses = []
auc_history = []
ap_history = []

# TRAINING
epochs = 300

for epoch in range(epochs):
    model.train()
    opt.zero_grad(set_to_none=True)

    _, z_train = model.encode(X_train.detach(), train_edge_index_seq)

    pos_logits = model.decode_logits(z_train, train_edge_index)

    neg_edge_index = negative_sampling(
        edge_index=train_edge_index,
        num_nodes=z_train.size(0),
        num_neg_samples=train_edge_index.size(1),
        method="sparse"
    ).to(device)

    neg_logits = model.decode_logits(z_train, neg_edge_index)

    logits = torch.cat([pos_logits, neg_logits], dim=0)
    labels = torch.cat([
        torch.ones_like(pos_logits),
        torch.zeros_like(neg_logits)
    ], dim=0)

    emb_penalty = 0.001 * z_train.pow(2).mean()
    loss = criterion(logits, labels) + emb_penalty

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    losses.append(float(loss.item()))

    del z_train, pos_logits, neg_logits, logits, labels, neg_edge_index, emb_penalty, loss

    if epoch % 5 == 0 or epoch == epochs - 1:
        model.eval()
        with torch.no_grad():
            _, z_eval = model.encode(X_train.detach(), train_edge_index_seq)

            pos_pred = model.decode_proba(z_eval, test_edge_index)

            neg_test_edge_index = negative_sampling(
                edge_index=test_edge_index,
                num_nodes=z_eval.size(0),
                num_neg_samples=test_edge_index.size(1),
                method="sparse"
            ).to(device)

            neg_pred = model.decode_proba(z_eval, neg_test_edge_index)

            preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
            labels_eval = torch.cat([
                torch.ones(pos_pred.size(0), device=device),
                torch.zeros(neg_pred.size(0), device=device)
            ]).cpu().numpy()

            auc = roc_auc_score(labels_eval, preds)
            ap = average_precision_score(labels_eval, preds)

            auc_history.append((epoch, float(auc)))
            ap_history.append((epoch, float(ap)))

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {losses[-1]:.4f} | "
            f"Test AUC: {auc:.4f} | "
            f"Test AP: {ap:.4f}"
        )

# FINAL EVALUATION
model.eval()
with torch.no_grad():
    h_train, z_train = model.encode(X_train.detach(), train_edge_index_seq)

    pos_pred = model.decode_proba(z_train, test_edge_index)

    neg_test_edge_index = negative_sampling(
        edge_index=test_edge_index,
        num_nodes=z_train.size(0),
        num_neg_samples=test_edge_index.size(1),
        method="sparse"
    ).to(device)

    neg_pred = model.decode_proba(z_train, neg_test_edge_index)

    preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
    labels_eval = torch.cat([
        torch.ones(pos_pred.size(0), device=device),
        torch.zeros(neg_pred.size(0), device=device)
    ]).cpu().numpy()

    auc = roc_auc_score(labels_eval, preds)
    ap = average_precision_score(labels_eval, preds)

z = z_train.detach().cpu()
z_norm = z / (z.norm(dim=1, keepdim=True) + 1e-8)

np.save(os.path.join(DATA_DIR, "embeddings", "z_2024.npy"), z.numpy())

print("\nFINAL MODEL PERFORMANCE:")
print(f"AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"Positive score max: {float(pos_pred.max()):.4f}")
print(f"Negative score max: {float(neg_pred.max()):.4f}")
print(f"Positive score mean: {float(pos_pred.mean()):.4f}")
print(f"Negative score mean: {float(neg_pred.mean()):.4f}")

BASE_DIR: /Users/carlos38/Desktop/Profesional/Master/GNN_course/EU_collabo_project
DATA_DIR: /Users/carlos38/Desktop/Profesional/Master/GNN_course/EU_collabo_project/data/processed
Using device: cpu
Train years: [1981, 1983, 1984, 1985, 1986, 1987, 1988, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Test year: 2024
X_train shape: (41, 4433, 96)
Using identity applicant mapping fallback for year 1981
Year 1981: reconstructed collaboration edges are empty
Year 1981: graph edges = (2, 4433)
Using identity applicant mapping fallback for year 1983
Year 1983: reconstructed collaboration edges are empty
Year 1983: graph edges = (2, 4433)
Using identity applicant mapping fallback for year 1984
Year 1984: graph edges = (2, 4436)
Using identity applicant mapping fallback for year 1985
Year 1985: graph edges = (2, 4436)
Using identity applic

In [198]:
print("\nFINAL MODEL PERFORMANCE:")
print(f"AUC: {auc:.4f}, AP: {ap:.4f}")


FINAL MODEL PERFORMANCE:
AUC: 0.9083, AP: 0.8703


In [ ]:
# SAVE METRICS
from sklearn.cluster import KMeans
def to_py(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, torch.Tensor):
        if x.numel() == 1:
            return x.item()
        return x.detach().cpu().tolist()
    if isinstance(x, list):
        return [to_py(v) for v in x]
    if isinstance(x, tuple):
        return [to_py(v) for v in x]
    if isinstance(x, dict):
        return {str(k): to_py(v) for k, v in x.items()}
    return x

metrics = {
    "losses": [float(v) for v in losses],
    "auc_history": [{"epoch": int(e), "auc": float(v)} for e, v in auc_history],
    "ap_history": [{"epoch": int(e), "ap": float(v)} for e, v in ap_history],
    "final_auc": float(auc),
    "final_ap": float(ap),
    "train_years": [int(y) for y in train_years],
    "test_year": int(test_year),
    "num_train_edges": int(train_edge_index.size(1)),
    "num_test_edges": int(test_edge_index.size(1)),
    "score_diagnostics": {
        "positive_max": float(pos_pred.max().item()),
        "positive_mean": float(pos_pred.mean().item()),
        "negative_max": float(neg_pred.max().item()),
        "negative_mean": float(neg_pred.mean().item())
    },
    "model_config": {
        "predictor": "mlp",
        "temporal": "attention",
        "dropout": 0.3,
        "activation": "LeakyReLU",
        "hidden_dim": 64
    }
}

metrics = to_py(metrics)

with open(os.path.join(DATA_DIR, "predictions", "training_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# NODE METADATA
id_to_app = {i: app for app, i in global_apps.items()}

df_nodes = pd.DataFrame({
    "node_id": list(id_to_app.keys()),
    "applicant_name": list(id_to_app.values())
}).sort_values("node_id")

# EMBEDDINGS / CLUSTERS
k = 10
clusters = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(z.numpy())
df_nodes["cluster"] = clusters

df_nodes.to_csv(
    os.path.join(DATA_DIR, "metadata", "nodes_with_clusters.csv"),
    index=False
)

np.save(os.path.join(DATA_DIR, "embeddings", "z_2024.npy"), z.numpy())
np.save(os.path.join(DATA_DIR, "embeddings", "z_norm.npy"), z_norm.numpy())

print("Saved nodes metadata and embeddings")

temporal_clusters = [
    KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(h_train[t].cpu().numpy())
    for t in range(h_train.shape[0])
]
np.save(
    os.path.join(DATA_DIR, "embeddings", "temporal_clusters.npy"),
    np.array(temporal_clusters)
)

cluster_matrix = np.zeros((k, k), dtype=np.float32)
for i in range(k):
    for j in range(k):
        mask_i = (clusters == i)
        mask_j = (clusters == j)
        if mask_i.sum() > 0 and mask_j.sum() > 0:
            sim_block = (z_norm[mask_i] @ z_norm[mask_j].T).numpy()
            cluster_matrix[i, j] = sim_block.mean()

np.save(
    os.path.join(DATA_DIR, "predictions", "cluster_matrix.npy"),
    cluster_matrix
)

# KNOWN EDGES
known_edges = set()

for src, dst in train_edge_index.t().tolist():
    a, b = (src, dst) if src < dst else (dst, src)
    known_edges.add((a, b))

for src, dst in test_edge_index.t().tolist():
    a, b = (src, dst) if src < dst else (dst, src)
    known_edges.add((a, b))

# TOP-K PREDICTED LINKS
top_links = {}
num_nodes = z.size(0)
TOP_K = 50

with torch.no_grad():
    for i in range(num_nodes):
        all_targets = torch.arange(num_nodes, dtype=torch.long)
        mask = all_targets != i
        targets = all_targets[mask]

        pair_index = torch.stack([
            torch.full_like(targets, i),
            targets
        ], dim=0)

        scores = model.decode_proba(z, pair_index).view(-1)

        scored = []
        for tgt, score in zip(targets.tolist(), scores.tolist()):
            a, b = (i, tgt) if i < tgt else (tgt, i)
            if (a, b) in known_edges:
                continue

            scored.append({
                "target": int(tgt),
                "score": float(score)
            })

        scored = sorted(scored, key=lambda x: x["score"], reverse=True)[:TOP_K]
        top_links[str(i)] = scored

with open(os.path.join(DATA_DIR, "predictions", "top_links.json"), "w") as f:
    json.dump(top_links, f)

print("Saved top predicted links")

# FULL PROBABILITY MATRIX
prob_matrix = torch.zeros((num_nodes, num_nodes))

with torch.no_grad():
    for i in range(num_nodes):
        all_targets = torch.arange(num_nodes, dtype=torch.long)
        pair_index = torch.stack([
            torch.full_like(all_targets, i),
            all_targets
        ], dim=0)

        scores = model.decode_proba(z, pair_index).view(-1)
        prob_matrix[i] = scores.cpu()

np.save(
    os.path.join(DATA_DIR, "predictions", "prob_matrix.npy"),
    prob_matrix.numpy()
)

print("Saved full probability matrix")
print("\nPIPELINE COMPLETE (METAPATH EMBEDDINGS + GRU TEMPORALITY + CO-FILING LABELS)")

✅ Saved nodes metadata and embeddings
✅ Saved top predicted links
✅ Saved full probability matrix

✅ PIPELINE COMPLETE (METAPATH EMBEDDINGS + GRU TEMPORALITY + CO-FILING LABELS)


# Ablation study

In [ ]:
# ABLATION STUDY:
# predictor = dot vs mlp
# temporal aggregation = attention vs mean
# metapath attention = True vs False

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import pandas as pd
import json
import os
import itertools

from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.utils import negative_sampling

# PATHS
BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(os.path.join(DATA_DIR, "predictions"), exist_ok=True)

# SEED
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# CHECK REQUIRED OBJECTS
required_vars = ["X", "data_by_year", "applicant_to_id"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}")

global_apps = applicant_to_id

# METAPATH SLICES
# X was built as concatenation of:
# APA (32) + API (32) + APC (32) = 96 dims
META_FEAT_SLICES = {
    "APA": slice(0, 32),
    "API": slice(32, 64),
    "APC": slice(64, 96),
}

print("META_FEAT_SLICES:", META_FEAT_SLICES)

# MODEL
class TemporalLinkPredictor(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=64,
        use_attention=True,
        use_metapath_attention=False,
        predictor="dot",
        dropout=0.3,
        meta_feat_slices=None
    ):
        super().__init__()

        self.use_attention = use_attention
        self.use_metapath_attention = use_metapath_attention
        self.predictor = predictor
        self.dropout = dropout
        self.meta_feat_slices = meta_feat_slices

        if self.use_metapath_attention:
            if meta_feat_slices is None or len(meta_feat_slices) < 2:
                raise ValueError("Metapath attention requires valid META_FEAT_SLICES.")

            self.meta_names = list(meta_feat_slices.keys())

            self.meta_proj = nn.ModuleDict({
                name: nn.Linear(
                    meta_feat_slices[name].stop - meta_feat_slices[name].start,
                    hidden_dim
                )
                for name in self.meta_names
            })

            self.meta_att_proj = nn.Linear(hidden_dim, 1)
            gru_input_dim = hidden_dim

        else:
            self.input_proj = nn.Linear(in_dim, hidden_dim)
            gru_input_dim = hidden_dim

        self.gru = nn.GRU(
            input_size=gru_input_dim,
            hidden_size=hidden_dim,
            num_layers=1
        )

        if self.use_attention:
            self.att_proj = nn.Linear(hidden_dim, 1)

        if self.predictor == "mlp":
            self.mlp = nn.Sequential(
                nn.Linear(hidden_dim * 4, hidden_dim),
                nn.LeakyReLU(negative_slope=0.1),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            )

    def apply_metapath_attention(self, x_t):
        meta_embs = []
        meta_scores = []

        for name in self.meta_names:
            sl = self.meta_feat_slices[name]
            x_meta = x_t[:, sl]                       # [N, 32]
            h_meta = torch.tanh(self.meta_proj[name](x_meta))  # [N, H]
            s_meta = self.meta_att_proj(h_meta)      # [N, 1]

            meta_embs.append(h_meta)
            meta_scores.append(s_meta)

        H_meta = torch.stack(meta_embs, dim=1)       # [N, M, H]
        S_meta = torch.stack(meta_scores, dim=1)     # [N, M, 1]
        A_meta = torch.softmax(S_meta, dim=1)        # [N, M, 1]

        h = (H_meta * A_meta).sum(dim=1)             # [N, H]
        return h

    def encode(self, X):
        # X: [T, N, F]
        snapshot_list = []

        for t in range(X.size(0)):
            x_t = X[t]

            if self.use_metapath_attention:
                h_t = self.apply_metapath_attention(x_t)
            else:
                h_t = self.input_proj(x_t)

            h_t = torch.nan_to_num(h_t, nan=0.0, posinf=5.0, neginf=-5.0)
            snapshot_list.append(h_t)

        H_in = torch.stack(snapshot_list, dim=0)     # [T, N, H]
        h, _ = self.gru(H_in)                        # [T, N, H]

        if self.use_attention:
            att = torch.clamp(self.att_proj(h), -10, 10)
            scores = torch.softmax(att, dim=0)
            z = (h * scores).sum(dim=0)
        else:
            z = h.mean(dim=0)

        z = torch.nan_to_num(z, nan=0.0, posinf=5.0, neginf=-5.0)
        return h, z

    def decode_logits(self, z, edge_index):
        if edge_index.numel() == 0:
            return torch.empty(0, device=z.device)

        src, dst = edge_index
        s = z[src]
        d = z[dst]

        if self.predictor == "dot":
            return (s * d).sum(dim=1)

        feat = torch.cat([
            s,
            d,
            torch.abs(s - d),
            s * d
        ], dim=1)

        return self.mlp(feat).view(-1)

    def decode_proba(self, z, edge_index, clip=8.0):
        logits = self.decode_logits(z, edge_index)
        logits = torch.clamp(logits, -clip, clip)
        return torch.sigmoid(logits)

# TEMPORAL SPLIT
years = sorted(data_by_year.keys())
if len(years) < 2:
    raise ValueError("Need at least two years for temporal split.")

train_years = years[:-1]
test_year = years[-1]
train_T = len(train_years)

print("Train years:", train_years)
print("Test year:", test_year)

# NORMALIZATION
X = torch.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0).float()

X_train_window = X[:train_T]
mean = X_train_window.mean(dim=(0, 1), keepdim=True)
std = X_train_window.std(dim=(0, 1), keepdim=True)
std = torch.where(std < 1e-6, torch.ones_like(std), std)

X = (X - mean) / std
X = torch.nan_to_num(X, nan=0.0, posinf=5.0, neginf=-5.0)

# important: remove any old autograd history
X = X.detach().clone()
X_train = X[:train_T].detach().clone().to(device)

print("X_train shape:", tuple(X_train.shape))

# BUILD LABEL EDGES
train_edge_list = []

for y in train_years:
    year_app_map = infer_year_app_map(y, data_by_year[y], global_apps)
    global_edge = reconstruct_global_collab_edges_from_patents(
        year=y,
        data=data_by_year[y],
        app_map=year_app_map,
        global_map=global_apps
    )

    if global_edge.numel() > 0:
        train_edge_list.append(global_edge)

if len(train_edge_list) == 0:
    raise ValueError("No training collaboration edges were reconstructed from train years.")

train_edge_index = canonicalize_edge_index(torch.cat(train_edge_list, dim=1)).to(device)
if train_edge_index.numel() == 0:
    raise ValueError("Training edge index is empty after cleaning.")

test_app_map = infer_year_app_map(test_year, data_by_year[test_year], global_apps)
test_edge_index = reconstruct_global_collab_edges_from_patents(
    year=test_year,
    data=data_by_year[test_year],
    app_map=test_app_map,
    global_map=global_apps
)
test_edge_index = canonicalize_edge_index(test_edge_index).to(device)

if test_edge_index.numel() == 0:
    raise ValueError("No test collaboration edges were reconstructed for the final year.")

print("Train edges:", tuple(train_edge_index.shape))
print("Test edges:", tuple(test_edge_index.shape))

# TRAIN/EVAL FUNCTION
def run_experiment(
    X_train,
    train_edge_index,
    test_edge_index,
    predictor="dot",
    use_attention=True,
    use_metapath_attention=False,
    hidden_dim=64,
    dropout=0.3,
    epochs=100,
    lr=5e-4,
    weight_decay=1e-4,
    seed=42
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = TemporalLinkPredictor(
        in_dim=X_train.shape[-1],
        hidden_dim=hidden_dim,
        use_attention=use_attention,
        use_metapath_attention=use_metapath_attention,
        predictor=predictor,
        dropout=dropout,
        meta_feat_slices=META_FEAT_SLICES
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = -1.0
    best_ap = -1.0
    best_epoch = -1

    for epoch in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)

        _, z_train = model.encode(X_train)

        pos_logits = model.decode_logits(z_train, train_edge_index)

        neg_edge_index = negative_sampling(
            edge_index=train_edge_index,
            num_nodes=z_train.size(0),
            num_neg_samples=train_edge_index.size(1),
            method="sparse"
        ).to(device)

        neg_logits = model.decode_logits(z_train, neg_edge_index)

        logits = torch.cat([pos_logits, neg_logits], dim=0)
        labels = torch.cat([
            torch.ones_like(pos_logits),
            torch.zeros_like(neg_logits)
        ], dim=0)

        emb_penalty = 0.001 * z_train.pow(2).mean()
        loss = criterion(logits, labels) + emb_penalty

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            _, z_eval = model.encode(X_train)

            pos_pred = model.decode_proba(z_eval, test_edge_index)

            neg_test_edge_index = negative_sampling(
                edge_index=test_edge_index,
                num_nodes=z_eval.size(0),
                num_neg_samples=test_edge_index.size(1),
                method="sparse"
            ).to(device)

            neg_pred = model.decode_proba(z_eval, neg_test_edge_index)

            preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
            labels_eval = torch.cat([
                torch.ones(pos_pred.size(0), device=device),
                torch.zeros(neg_pred.size(0), device=device)
            ]).cpu().numpy()

            auc = roc_auc_score(labels_eval, preds)
            ap = average_precision_score(labels_eval, preds)

            if auc > best_auc:
                best_auc = float(auc)
                best_ap = float(ap)
                best_epoch = epoch + 1

        del z_train, pos_logits, neg_edge_index, neg_logits, logits, labels, emb_penalty, loss

    # final evaluation
    model.eval()
    with torch.no_grad():
        _, z_final = model.encode(X_train)

        pos_pred = model.decode_proba(z_final, test_edge_index)

        neg_test_edge_index = negative_sampling(
            edge_index=test_edge_index,
            num_nodes=z_final.size(0),
            num_neg_samples=test_edge_index.size(1),
            method="sparse"
        ).to(device)

        neg_pred = model.decode_proba(z_final, neg_test_edge_index)

        preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
        labels_eval = torch.cat([
            torch.ones(pos_pred.size(0), device=device),
            torch.zeros(neg_pred.size(0), device=device)
        ]).cpu().numpy()

        final_auc = roc_auc_score(labels_eval, preds)
        final_ap = average_precision_score(labels_eval, preds)

    return {
        "metapath_attention": use_metapath_attention,
        "dynamic_attention": use_attention,
        "predictor": predictor,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "epochs": epochs,
        "best_auc": float(best_auc),
        "best_ap": float(best_ap),
        "best_epoch": int(best_epoch),
        "final_auc": float(final_auc),
        "final_ap": float(final_ap),
    }

# ABLATION GRID
configs = list(itertools.product(
    [False, True],   # metapath attention
    [False, True],   # dynamic attention
    ["dot", "mlp"]   # predictor
))

results = []

print("\n=== ABLATION STUDY ===")
for use_meta, use_dyn, pred in configs:
    print(
        f"Running metapath_attention={use_meta} | "
        f"dynamic_attention={use_dyn} | "
        f"predictor={pred}"
    )

    out = run_experiment(
        X_train=X_train,
        train_edge_index=train_edge_index,
        test_edge_index=test_edge_index,
        predictor=pred,
        use_attention=use_dyn,
        use_metapath_attention=use_meta,
        hidden_dim=64,
        dropout=0.3,
        epochs=100,
        lr=5e-4,
        weight_decay=1e-4,
        seed=42
    )

    results.append(out)

    print(
        f" -> final AUC: {out['final_auc']:.4f} | "
        f"final AP: {out['final_ap']:.4f} | "
        f"best AUC: {out['best_auc']:.4f} @ epoch {out['best_epoch']}"
    )

# RESULTS TABLE
results_df = pd.DataFrame(results).sort_values(
    ["final_auc", "final_ap"],
    ascending=False
).reset_index(drop=True)

print("\n=== ABLATION RESULTS ===")
print(results_df)

results_df.to_csv(
    os.path.join(DATA_DIR, "predictions", "ablation_metapath_dynamic_predictor.csv"),
    index=False
)

with open(os.path.join(DATA_DIR, "predictions", "ablation_metapath_dynamic_predictor.json"), "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved ablation results.")

Using device: cpu
META_FEAT_SLICES: {'APA': slice(0, 32, None), 'API': slice(32, 64, None), 'APC': slice(64, 96, None)}
Train years: [1981, 1983, 1984, 1985, 1986, 1987, 1988, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Test year: 2024
X_train shape: (41, 4339, 96)
Using identity applicant mapping fallback for year 1981
Year 1981: reconstructed collaboration edges are empty
Using identity applicant mapping fallback for year 1983
Year 1983: reconstructed collaboration edges are empty
Using identity applicant mapping fallback for year 1984
Using identity applicant mapping fallback for year 1985
Using identity applicant mapping fallback for year 1986
Using identity applicant mapping fallback for year 1987
Using identity applicant mapping fallback for year 1988
Year 1988: reconstructed collaboration edges are empty
Using identity ap

In [38]:
results_df

,metapath_attention,dynamic_attention,predictor,hidden_dim,dropout,epochs,best_auc,best_ap,best_epoch,final_auc,final_ap
0,False,True,mlp,64,0.3,100,0.775896,0.741547,96,0.769635,0.737679
1,False,False,mlp,64,0.3,100,0.766663,0.732571,99,0.758717,0.725488
2,True,False,mlp,64,0.3,100,0.743187,0.713792,100,0.737190,0.708912
3,True,True,mlp,64,0.3,100,0.740091,0.713079,100,0.734864,0.707074
4,True,False,dot,64,0.3,100,0.564817,0.557820,100,0.561028,0.556766
5,True,True,dot,64,0.3,100,0.562622,0.554926,100,0.558828,0.553859
6,False,False,dot,64,0.3,100,0.562010,0.561735,55,0.550883,0.548908
7,False,True,dot,64,0.3,100,0.562438,0.563879,55,0.549249,0.551884
